# Design of experiments (DOE): SFT hyperparameter sweeps

This notebook is a **thin demo**: every step calls into the tested
`geap_tuning` package. See
[`docs/notes/doe-and-visualization.md`](../docs/notes/doe-and-visualization.md).

A **DOE** declares a hyperparameter grid once and runs it as a unit:
`run_sweep` expands the grid into one run per point (each with a deterministic
display name), **reuses** a finished job with that name instead of re-tuning,
scores each tuned endpoint offline, and logs it to **Vertex AI Experiments** so
the runs compare side by side.

**Where each plottable metric comes from:**

| Source | Granularity | Fetch |
|---|---|---|
| `aggregate_results` rows / `experiment_dataframe` | one value/metric/run | programmatic, TensorBoard-free |
| `collect_checkpoint_curve` | value/epoch/run | programmatic (per-checkpoint eval) |
| Layer-1 Monitor loss curves | value/step/run | **console-only — not via the SDK** |

Scope here is **SFT only**. Charts need the optional viz group
(`uv sync --group viz`).

> **Requires live GCP and incurs tuning cost.** Have a real `.env` and
> `gcloud auth` in place; keep the tuning/Experiments region aligned.


In [ ]:
from geap_tuning.config import genai_client, load_config
from geap_tuning.doe import SweepConfig

EXPERIMENT_NAME = "geap-doe-sft"
SWEEP = SweepConfig(
    name="cheap",
    base_model="gemini-2.5-flash-lite",
    grid={"epochs": [1, 2], "adapter_size": [4, 8]},  # 4 runs
)

cfg = load_config()
client = genai_client(cfg)
cfg


## 1. Build and stage the SFT dataset

Same deterministic support-intent splits as the SFT notebook; the test split is
held out locally for offline scoring.

In [ ]:
from geap_tuning.gcs import upload_file
from geap_tuning.sft.data import SUPPORT_TICKETS, build_records, build_sft_dataset, split_dataset

paths = build_sft_dataset("../datasets/sft_support_intent")
train_uri = upload_file(paths["train"], f"{cfg.bucket}/doe_sft/train.jsonl")
val_uri = upload_file(paths["val"], f"{cfg.bucket}/doe_sft/val.jsonl")

_, _, test_pairs = split_dataset(SUPPORT_TICKETS)
test_records = build_records(test_pairs)
train_uri, val_uri


## 2. Point Vertex AI Experiments at the tuning region

Call `init_experiment` **before** `run_sweep` so each run's params + metrics
attach to it (no TensorBoard needed for summary metrics).

In [ ]:
from geap_tuning.experiments import init_experiment

init_experiment(EXPERIMENT_NAME, project=cfg.project, location=cfg.location)


## 3. Define the offline scorer

`run_sweep` calls `evaluate_fn(endpoint)` for each run; here we score on the
held-out test split.

In [ ]:
from geap_tuning.inference import generate
from geap_tuning.sft.evaluate import run_eval


def evaluate_fn(endpoint: str) -> dict:
    return run_eval(
        test_records,
        predict_fn=lambda user_text, e=endpoint: generate(client, e, user_text),
    )


## 4. Run the sweep

Each grid point reuses a matching job if one exists (cost control), else launches
a `gemini-2.5-flash-lite` SFT job, waits, scores, and logs to Experiments.

In [ ]:
from geap_tuning.doe import run_sweep

results = run_sweep(
    client,
    SWEEP,
    train_uri=train_uri,
    val_uri=val_uri,
    evaluate_fn=evaluate_fn,
    experiment=EXPERIMENT_NAME,
)
[(r.spec.name, round(r.metrics["accuracy"], 3), "reused" if r.reused else "launched") for r in results]


## 5. Aggregate and pick the winner

`aggregate_results` flattens the runs into one comparable row each — the single
shaped structure that feeds both the table and the charts.

In [ ]:
from geap_tuning.doe import aggregate_results, select_best_run

rows = aggregate_results(results)
best = select_best_run({r.spec.name: r.metrics for r in results})
print("best run (accuracy):", best)
rows


## 6. Compare runs visually

Requires the viz group (`uv sync --group viz`). `plot_grouped_metric_bars` draws
one cluster per run with a bar per metric.

In [ ]:
from geap_tuning.viz import plot_grouped_metric_bars

plot_grouped_metric_bars(rows)


## 7. The same runs, read back from Experiments

`experiment_dataframe` returns a pandas table matching **Agent Platform Studio →
Experiments** — [`11_multi_run_viz.ipynb`](11_multi_run_viz.ipynb) charts this
directly with no tuning cost.

In [ ]:
from geap_tuning.experiments import experiment_dataframe

experiment_dataframe(EXPERIMENT_NAME)


## Next steps

Extend the grid (e.g. add `learning_rate_multiplier`) or read the tracked runs
back with zero tuning cost in [`11_multi_run_viz.ipynb`](11_multi_run_viz.ipynb).
DPO/RLFT sweeps are a documented follow-up — see
[`docs/notes/doe-and-visualization.md`](../docs/notes/doe-and-visualization.md).
